# 🎵 Chinook Music Store Database Analysis

This notebook demonstrates Calliope AI using the **Chinook** database - a digital music store sample database.

## 📊 Database Overview

The Chinook database represents a digital music store with:
- **Artists & Albums**: Music catalog
- **Tracks**: Individual songs with genre, media type, and pricing
- **Customers**: Customer information from different countries
- **Employees**: Sales support staff
- **Invoices**: Sales transactions
- **Playlists**: Curated track collections

## 🔗 Real Datasource Connection

**Datasource ID**: `chinook`  
**Dialect**: PostgreSQL  
**Pre-configured** on the Calliope API server (port 5000)

## ⚙️ Configuration

**Run this cell first!** Sets up the Calliope API endpoint.

In [ ]:
import os

# Configure Calliope Data Agent API endpoint
# Change this if your setup is different
API_BASE_URL = 'http://localhost:5000'  # Local development
# API_BASE_URL = 'http://data-agent:5000'  # Docker/JupyterHub

os.environ['CALLIOPE_API_BASE_URL'] = API_BASE_URL
print(f"✅ Calliope API configured: {API_BASE_URL}")

## ✅ Verify Connection

In [ ]:
import requests

response = requests.get('http://localhost:5000/api/datasources')
datasources = response.json()['datasources']
chinook = [ds for ds in datasources if ds['id'] == 'chinook'][0]

print("✅ Connected to Chinook database!")
print(f"Datasource: {chinook['name']}")
print(f"Dialect: {chinook['dialect']}")

## 🔍 Explore Database Schema

In [ ]:
%%calliope run-sql chinook
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public' 
ORDER BY table_name

## 🎵 Music Catalog Analysis

In [ ]:
%%calliope ask-sql chinook
How many artists, albums, and tracks are in the database?

In [ ]:
%%calliope ask-sql chinook
What are the top 10 artists by number of tracks?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Show the distribution of tracks by genre

In [ ]:
%%calliope ask-sql chinook
Which album has the most tracks?

## 💰 Sales Revenue Analysis

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
What is the total sales revenue?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Show revenue by country (top 10 countries)

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Which are the top 10 best-selling tracks by revenue?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
What is the revenue breakdown by genre?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Show monthly sales trends over time

## 👥 Customer Analysis

In [ ]:
%%calliope ask-sql chinook
How many customers do we have in total?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Who are the top 10 customers by total spending?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Show customer distribution by country

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Calculate average customer lifetime value

## 🎸 Genre Preferences Analysis

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
What are the most popular genres by number of purchases?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Show genre preferences by country (top 5 countries)

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Which genres have the highest average track price?

## 👔 Employee Sales Performance

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Show sales performance by employee (sales support agents)

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Which employee has the most customers assigned?

## 🎤 Artist Performance Analysis

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Who are the top 10 artists by sales revenue?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Which artists have the most albums in the catalog?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Find the best-selling album

## 🚀 Advanced Analytics

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Calculate the customer purchase frequency (average days between purchases)

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Find customers who have purchased from multiple genres (diverse listeners)

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic --to-ai --model claude3
Analyze the relationship between track length and sales. Do shorter or longer tracks sell better? 
Provide insights and recommendations for the music catalog strategy.

## 📋 Playlist Analysis

In [ ]:
%%calliope ask-sql chinook
How many playlists exist and what are their names?

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Which playlist has the most tracks?

## 💿 Media Type Analysis

In [ ]:
%%calliope ask-sql chinook
Show the distribution of tracks by media type

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Which media type generates the most revenue?

## 📊 Customer Cohort Analysis

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Group customers by their first purchase year and show their total lifetime value

## 💻 Complex SQL Queries

For advanced users who want direct SQL control:

In [ ]:
%%calliope run-sql chinook
SELECT 
    g.name AS genre,
    COUNT(DISTINCT il.invoice_id) AS num_purchases,
    COUNT(DISTINCT il.track_id) AS unique_tracks_sold,
    SUM(il.quantity) AS total_quantity,
    ROUND(SUM(il.unit_price * il.quantity)::numeric, 2) AS total_revenue,
    ROUND(AVG(il.unit_price)::numeric, 2) AS avg_price
FROM genre g
JOIN track t ON g.genre_id = t.genre_id
JOIN invoice_line il ON t.track_id = il.track_id
GROUP BY g.genre_id, g.name
ORDER BY total_revenue DESC

In [ ]:
%%calliope run-sql chinook
WITH customer_stats AS (
    SELECT 
        c.customer_id,
        c.first_name || ' ' || c.last_name AS customer_name,
        c.country,
        COUNT(DISTINCT i.invoice_id) AS num_purchases,
        SUM(i.total) AS total_spent,
        MIN(i.invoice_date) AS first_purchase,
        MAX(i.invoice_date) AS last_purchase
    FROM customer c
    JOIN invoice i ON c.customer_id = i.customer_id
    GROUP BY c.customer_id, customer_name, c.country
)
SELECT 
    customer_name,
    country,
    num_purchases,
    ROUND(total_spent::numeric, 2) AS total_spent,
    ROUND((total_spent / num_purchases)::numeric, 2) AS avg_purchase_value,
    first_purchase::date,
    last_purchase::date
FROM customer_stats
ORDER BY total_spent DESC
LIMIT 20

## 🤔 Generate Insights

In [ ]:
%%calliope followup-questions chinook
What additional business questions should I explore with the Chinook music store data?

## 📊 Key Insights

Using Calliope with Chinook database, we explored:
- Music catalog analytics (artists, albums, tracks, genres)
- Sales revenue analysis by geography and time
- Customer segmentation and lifetime value
- Genre preferences and trends
- Employee performance metrics
- Artist and album popularity

## 🎯 Next Steps

Explore other real datasources:
- **Calliope-RealData-Sakila.ipynb** - DVD rental store
- **Calliope-RealData-Northwind.ipynb** - Business operations
- **Calliope-RealData-Employees.ipynb** - HR analytics